# 02. Pollution & DSC Scoring

**Phase 3**: DQ4AI polluter로 오염 생성 → 각 오염 데이터에 DSC 점수 계산

**사전 조건**: `01_setup_and_baseline.ipynb` 실행 완료

---

## 0. 환경 설정

In [1]:
# ============================================================
# 0-1. Google Drive 마운트 & 경로
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/capstone/dsc'
RAW_DIR = f'{BASE}/data/raw'
POLLUTED_DIR = f'{BASE}/data/polluted'
RESULTS_DIR = f'{BASE}/results'
DQ4AI_DIR = f'{BASE}/dq4ai'

# DQ4AI를 import 경로에 추가
sys.path.insert(0, DQ4AI_DIR)

print('환경 설정 완료')

Mounted at /content/drive
환경 설정 완료


In [2]:
# ============================================================
# 0-2. 데이터셋 메타데이터 (01과 동일)
# ============================================================
DATASETS = {
    'TelcoCustomerChurn': {
        'path': f'{RAW_DIR}/TelcoCustomerChurn.csv',
        'target': 'Churn',
        'numerical_cols': ['tenure', 'MonthlyCharges', 'TotalCharges'],
        'categorical_cols': [
            'gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod'
        ],
        'placeholder_numerical': -1,
        'placeholder_categorical': {
            'Contract': 'empty', 'Dependents': 'empty', 'DeviceProtection': 'empty',
            'InternetService': 'empty', 'MultipleLines': 'empty', 'OnlineBackup': 'empty',
            'OnlineSecurity': 'empty', 'PaperlessBilling': 'empty', 'Partner': 'empty',
            'PaymentMethod': 'empty', 'PhoneService': 'empty', 'SeniorCitizen': -1,
            'StreamingMovies': 'empty', 'StreamingTV': 'empty', 'TechSupport': 'empty',
            'gender': 'empty'
        },
    },
    'SouthGermanCredit': {
        'path': f'{RAW_DIR}/SouthGermanCredit.csv',
        'target': 'credit_risk',
        'numerical_cols': ['duration', 'amount', 'age'],
        'categorical_cols': [
            'status', 'credit_history', 'purpose', 'savings',
            'employment_duration', 'installment_rate', 'personal_status_sex',
            'other_debtors', 'present_residence', 'property',
            'other_installment_plans', 'housing', 'number_credits',
            'job', 'people_liable', 'telephone', 'foreign_worker'
        ],
        'placeholder_numerical': -1,
        'placeholder_categorical': 'missing',
    },
    'letter': {
        'path': f'{RAW_DIR}/letter.csv',
        'target': 'lettr',
        'numerical_cols': [
            'x-box', 'y-box', 'width', 'high', 'onpix', 'x-bar',
            'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br',
            'x-ege', 'xegvy', 'y-ege', 'yegvx'
        ],
        'categorical_cols': [],
        'placeholder_numerical': -1,
        'placeholder_categorical': None,
    },
}

POLLUTION_LEVELS = [0.1, 0.25, 0.5, 0.75]
RANDOM_SEED = 42

print(f'데이터셋: {list(DATASETS.keys())}')
print(f'오염 강도: {POLLUTION_LEVELS}')

데이터셋: ['TelcoCustomerChurn', 'SouthGermanCredit', 'letter']
오염 강도: [0.1, 0.25, 0.5, 0.75]


## 1. DQ4AI Polluter 래핑

DQ4AI polluter를 직접 인스턴스화하여 사용한다.  
pandas 호환성 문제(`df.append` deprecated)에 대비하여 래퍼를 만든다.

In [3]:
# ============================================================
# 1-1. DQ4AI polluter import 및 pandas 호환 패치
# ============================================================

# pandas 2.x 호환: df.append → pd.concat 패치
if not hasattr(pd.DataFrame, 'append'):
    def _df_append(self, other, ignore_index=False, **kwargs):
        return pd.concat([self, other], ignore_index=ignore_index)
    pd.DataFrame.append = _df_append
    print('pandas 2.x 호환 패치 적용')

from polluters.completeness_polluter import CompletenessPolluter
from polluters.uniqueness_polluter import UniquenessPolluter
from polluters.feature_accuracy_polluter import FeatureAccuracyPolluter
from polluters.consistent_representation_polluter import ConsistentRepresentationPolluter
from polluters.classbalance import ClassBalancePolluter

print('DQ4AI polluter import 완료')

pandas 2.x 호환 패치 적용
DQ4AI polluter import 완료


In [4]:
# ============================================================
# 1-2. Polluter 인스턴스 생성 함수
# ============================================================

def create_polluters(ds_name, meta, df):
    """데이터셋별로 (polluter_name, level, polluter_instance) 리스트 생성."""
    results = []
    target = meta['target']
    num_cols = meta['numerical_cols']
    cat_cols = meta['categorical_cols']
    ph_num = meta['placeholder_numerical']
    ph_cat = meta['placeholder_categorical']

    for level in POLLUTION_LEVELS:
        # --- Completeness ---
        results.append(('completeness', level, CompletenessPolluter(
            pollution_percentages=level,
            target_feature=target,
            placeholder_numerical=ph_num,
            placeholder_categorical=ph_cat if ph_cat is not None else 'empty',
            numerical_cols=num_cols,
            categorical_cols=cat_cols,
            random_seed=RANDOM_SEED,
        )))

        # --- Uniqueness ---
        # level 0.1→factor1.5, 0.25→2.0, 0.5→3.0, 0.75→4.0
        factor_map = {0.1: 1.5, 0.25: 2.0, 0.5: 3.0, 0.75: 4.0}
        results.append(('uniqueness', level, UniquenessPolluter(
            duplicate_factor=factor_map[level],
            distribution_function_name='same',
            distribution_function_parameters={},
            target_feature=target,
            random_seed=RANDOM_SEED,
        )))

        # --- FeatureAccuracy ---
        results.append(('feature_accuracy', level, FeatureAccuracyPolluter(
            pollution_levels=level,
            categorical_cols=cat_cols,
            numerical_cols=num_cols,
            random_seed=RANDOM_SEED,
        )))

        # --- ConsistentRepresentation (범주형 있는 데이터만) ---
        if cat_cols:
            num_of_repr = {
                col: {val: 2 for val in df[col].dropna().unique()}
                for col in cat_cols if col in df.columns
            }
            results.append(('consistent_repr', level, ConsistentRepresentationPolluter(
                random_seed=RANDOM_SEED,
                percentage_polluted_rows=level,
                num_pollutable_columns=len(cat_cols),
                number_of_representations=num_of_repr,
            )))

        # --- ClassBalance ---
        results.append(('class_balance', level, ClassBalancePolluter(
            imbalance_level=level,
            target_column=target,
            n_samples=len(df),
            random_seed=RANDOM_SEED,
        )))

    return results


print('Polluter 팩토리 정의 완료')

Polluter 팩토리 정의 완료


## 2. 오염 실행 & DSC 점수 계산

In [5]:
# ============================================================
# 2-1. DSC 스코어링 엔진 정의 (v3 — outlier reference 고정 + value_accuracy 신설)
# ============================================================
import numpy as np
import re
from scipy import stats as sp_stats


def calc_completeness(df, target_col, placeholder_numerical=-1, placeholder_categorical='empty'):
    """결측치 + placeholder 비율. 1=완전, 0=전부 결측."""
    feature_df = df.drop(columns=[target_col], errors='ignore')
    total_cells = feature_df.shape[0] * feature_df.shape[1]
    if total_cells == 0:
        return 1.0
    missing_count = feature_df.isnull().sum().sum()
    if placeholder_numerical is not None:
        for col in feature_df.select_dtypes(include=[np.number]).columns:
            missing_count += (feature_df[col] == placeholder_numerical).sum()
    if placeholder_categorical is not None:
        for col in feature_df.select_dtypes(include=['object', 'category']).columns:
            ph = (placeholder_categorical.get(col, 'empty')
                  if isinstance(placeholder_categorical, dict)
                  else placeholder_categorical)
            missing_count += (feature_df[col].astype(str) == str(ph)).sum()
    return 1.0 - (missing_count / total_cells)


def calc_uniqueness(df, target_col):
    """중복 행 비율. 1=전부 유일."""
    n = len(df)
    if n <= 1:
        return 1.0
    return 1.0 - (df.duplicated().sum() / n)


def calc_validity(df, target_col, numerical_cols, categorical_cols):
    """타입/형식 유효성."""
    scores = []
    for col in numerical_cols:
        if col not in df.columns:
            continue
        converted = pd.to_numeric(df[col], errors='coerce')
        total = len(df[col].dropna())
        scores.append(converted.notna().sum() / total if total > 0 else 1.0)
    for col in categorical_cols:
        if col not in df.columns:
            continue
        s = df[col].dropna().astype(str)
        if len(s) == 0:
            scores.append(1.0)
            continue
        valid = s.apply(lambda x: 0 < len(x.strip()) < 200)
        scores.append(valid.mean())
    return float(np.mean(scores)) if scores else 1.0


def calc_consistency(df, target_col, categorical_cols, reference_df=None,
                     placeholder_categorical='empty'):
    """카테고리 표현 일관성.
    v3.2: reference 있을 때 — reference에 없던 새 표현이 차지하는 행 비율의 보수.
                              placeholder는 결측 신호이므로 제외(completeness와 분리).
          reference 없을 때 — 기존 \'-숫자\' 정규식 fallback."""
    if not categorical_cols:
        return 1.0
    scores = []
    for col in categorical_cols:
        if col not in df.columns:
            continue
        cur_vals = df[col].dropna().astype(str)
        if len(cur_vals) == 0:
            scores.append(1.0); continue
        ph = None
        if placeholder_categorical is not None:
            ph = (str(placeholder_categorical.get(col, 'empty'))
                  if isinstance(placeholder_categorical, dict)
                  else str(placeholder_categorical))
            cur_vals = cur_vals[cur_vals != ph]
        if len(cur_vals) == 0:
            scores.append(1.0); continue
        if reference_df is not None and col in reference_df.columns:
            ref_vals = reference_df[col].dropna().astype(str)
            if ph is not None:
                ref_vals = ref_vals[ref_vals != ph]
            ref_set = set(ref_vals.unique())
            new_row_ratio = (~cur_vals.isin(ref_set)).mean()
            scores.append(1.0 - float(new_row_ratio))
        else:
            has_suffix = cur_vals.apply(lambda x: bool(re.search(r'-\d+$', x)))
            scores.append(1.0 - float(has_suffix.mean()))
    return float(np.mean(scores)) if scores else 1.0


def calc_outlier_ratio(df, target_col, numerical_cols, reference_df=None):
    """IQR 기반 outlier가 아닌 비율.
    v3: reference_df의 IQR을 고정 기준으로 사용 → 노이즈가 IQR을 부풀려 outlier가
    줄어드는 자기참조 함정 회피."""
    if not numerical_cols:
        return 1.0
    scores = []
    for col in numerical_cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(s) < 4:
            scores.append(1.0)
            continue
        if reference_df is not None and col in reference_df.columns:
            ref = pd.to_numeric(reference_df[col], errors='coerce').dropna()
            if len(ref) >= 4:
                q1, q3 = ref.quantile(0.25), ref.quantile(0.75)
            else:
                q1, q3 = s.quantile(0.25), s.quantile(0.75)
        else:
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            scores.append(1.0)
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_count = ((s < lower) | (s > upper)).sum()
        scores.append(1.0 - outlier_count / len(s))
    return float(np.mean(scores)) if scores else 1.0


def calc_class_balance(df, target_col):
    """클래스 균형 — 최소 비율 / 이상 비율."""
    counts = df[target_col].value_counts()
    n_classes = len(counts)
    if n_classes <= 1:
        return 1.0
    min_ratio = counts.min() / counts.sum()
    ideal_ratio = 1.0 / n_classes
    return min(min_ratio / ideal_ratio, 1.0)


def calc_feature_correlation(df, target_col, numerical_cols, threshold=0.95):
    """고상관(>threshold) 피처 쌍이 없는 비율."""
    cols = [c for c in numerical_cols if c in df.columns]
    if len(cols) < 2:
        return 1.0
    num_df = df[cols].apply(pd.to_numeric, errors='coerce')
    corr = num_df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    total_pairs = upper.size - upper.isna().sum().sum()
    if total_pairs == 0:
        return 1.0
    high_corr_pairs = (upper > threshold).sum().sum()
    return 1.0 - (high_corr_pairs / total_pairs)


def calc_value_accuracy(df, target_col, numerical_cols, categorical_cols, reference_df=None):
    """값 정확성 — reference 분포 대비 drift.
    수치형: 1 - KS statistic. 범주형: 1 - Total Variation Distance.
    reference 없으면 1.0 (지표 비활성)."""
    if reference_df is None:
        return 1.0
    scores = []
    for col in numerical_cols:
        if col not in df.columns or col not in reference_df.columns:
            continue
        ref = pd.to_numeric(reference_df[col], errors='coerce').dropna()
        cur = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(ref) < 5 or len(cur) < 5:
            scores.append(1.0)
            continue
        ks_stat, _ = sp_stats.ks_2samp(ref, cur)
        scores.append(1.0 - float(ks_stat))
    for col in categorical_cols:
        if col not in df.columns or col not in reference_df.columns:
            continue
        ref_counts = reference_df[col].astype(str).value_counts(normalize=True)
        cur_counts = df[col].astype(str).value_counts(normalize=True)
        all_cats = ref_counts.index.union(cur_counts.index)
        ref_p = ref_counts.reindex(all_cats, fill_value=0)
        cur_p = cur_counts.reindex(all_cats, fill_value=0)
        tvd = 0.5 * (ref_p - cur_p).abs().sum()
        scores.append(1.0 - float(tvd))
    return float(np.mean(scores)) if scores else 1.0


DEFAULT_WEIGHTS = {
    'completeness': 0.20, 'uniqueness': 0.15, 'validity': 0.05,
    'consistency': 0.10, 'outlier_ratio': 0.05,
    'class_balance': 0.10, 'feature_correlation': 0.05,
    'value_accuracy': 0.30,
}


def compute_dsc(df, target_col, numerical_cols, categorical_cols,
                weights=None,
                placeholder_numerical=-1,
                placeholder_categorical='empty',
                reference_df=None):
    """DSC 점수(0~100) + 등급 + 지표별 점수.
    v3: value_accuracy 신설, outlier_ratio reference 고정, 가중치 재배분.
    reference_df: 베이스라인 데이터(보통 clean df). 베이스라인 자체 측정 시 자기 자신을 넘김."""
    w = weights or DEFAULT_WEIGHTS
    metrics = {
        'completeness':        calc_completeness(df, target_col, placeholder_numerical, placeholder_categorical),
        'uniqueness':          calc_uniqueness(df, target_col),
        'validity':            calc_validity(df, target_col, numerical_cols, categorical_cols),
        'consistency':         calc_consistency(df, target_col, categorical_cols, reference_df=reference_df, placeholder_categorical=placeholder_categorical),
        'outlier_ratio':       calc_outlier_ratio(df, target_col, numerical_cols, reference_df=reference_df),
        'class_balance':       calc_class_balance(df, target_col),
        'feature_correlation': calc_feature_correlation(df, target_col, numerical_cols),
        'value_accuracy':      calc_value_accuracy(df, target_col, numerical_cols, categorical_cols, reference_df=reference_df),
    }
    score = sum(metrics[k] * w[k] for k in w) * 100
    if score >= 90:   grade = 'A'
    elif score >= 75: grade = 'B'
    elif score >= 60: grade = 'C'
    else:             grade = 'D'
    return {'score': round(score, 2), 'grade': grade,
            **{k: round(v, 4) for k, v in metrics.items()}}


print('DSC 스코어링 엔진 v3 정의 완료')
print('  변경: value_accuracy 신설, outlier_ratio reference 고정, 가중치 재배분')

DSC 스코어링 엔진 v2 정의 완료
  변경: completeness(placeholder감지), consistency(행비율), class_balance(최소비율)


In [6]:
# ============================================================
# 2-2. Split → Train 폴루션 → DSC 계산 (단일 트랙)
# ============================================================
# split-first 원칙: clean → train/test split → train에만 polluter 적용
# DSC와 ML이 동일한 train_polluted를 사용 (P6 해결).
# reference_df = df_train_clean → outlier·value_accuracy의 분포 거리 비교 기준.
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder as _LE
from time import time

TRAIN_DIR = f'{BASE}/data/train_polluted'
TEST_DIR = f'{BASE}/data/test_clean'
ML_SPLIT_SEED = 1
ML_TEST_SIZE = 0.2

dsc_rows = []
train_total = 0
error_log = []

total_start = time()

for ds_name, meta in DATASETS.items():
    print()
    print('=' * 60)
    print(f'데이터셋: {ds_name}')
    print('=' * 60)

    df_clean = pd.read_csv(meta['path'])
    if 'TotalCharges' in df_clean.columns:
        df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0)

    # --- Split (고정 seed, stratified) ---
    target = meta['target']
    _le = _LE()
    y_encoded = _le.fit_transform(df_clean[target].astype(str))
    train_idx, test_idx = train_test_split(
        np.arange(len(df_clean)), test_size=ML_TEST_SIZE,
        random_state=ML_SPLIT_SEED, stratify=y_encoded
    )
    df_train_clean = df_clean.iloc[train_idx].reset_index(drop=True)
    df_test_clean = df_clean.iloc[test_idx].reset_index(drop=True)

    # --- Test 저장 (데이터셋당 1회) ---
    os.makedirs(TEST_DIR, exist_ok=True)
    df_test_clean.to_csv(f'{TEST_DIR}/{ds_name}_test.csv', index=False)
    print(f'  Test 저장: {len(df_test_clean)}행')

    # --- Baseline (train clean) 저장 + DSC ---
    baseline_dir = f'{TRAIN_DIR}/{ds_name}/none_0'
    os.makedirs(baseline_dir, exist_ok=True)
    df_train_clean.to_csv(f'{baseline_dir}/train_data.csv', index=False)
    train_total += 1

    baseline_dsc = compute_dsc(
        df_train_clean,
        target_col=meta['target'],
        numerical_cols=meta['numerical_cols'],
        categorical_cols=meta['categorical_cols'],
        placeholder_numerical=meta.get('placeholder_numerical', -1),
        placeholder_categorical=meta.get('placeholder_categorical', 'empty'),
        reference_df=df_train_clean,
    )
    dsc_rows.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0, **baseline_dsc})
    print(f'  baseline (train clean, {len(df_train_clean)}행) → DSC={baseline_dsc["score"]:6.2f} ({baseline_dsc["grade"]})')

    # --- 각 polluter를 train에 적용 + DSC 계산 ---
    polluter_list = create_polluters(ds_name, meta, df_train_clean)
    for polluter_name, level, polluter in polluter_list:
        label = f'{ds_name}/{polluter_name}_{int(level*100)}%'
        try:
            t0 = time()
            polluted_train = polluter.pollute(df_train_clean.copy())
            elapsed = time() - t0

            save_dir = f'{TRAIN_DIR}/{ds_name}/{polluter_name}_{int(level*100)}'
            os.makedirs(save_dir, exist_ok=True)
            polluted_train.to_csv(f'{save_dir}/train_data.csv', index=False)
            train_total += 1

            dsc_result = compute_dsc(
                polluted_train,
                target_col=meta['target'],
                numerical_cols=meta['numerical_cols'],
                categorical_cols=meta['categorical_cols'],
                placeholder_numerical=meta.get('placeholder_numerical', -1),
                placeholder_categorical=meta.get('placeholder_categorical', 'empty'),
                reference_df=df_train_clean,
            )
            dsc_rows.append({'dataset': ds_name, 'polluter': polluter_name, 'level': level, **dsc_result})
            print(f'  {label:45s} → DSC={dsc_result["score"]:6.2f} ({dsc_result["grade"]})  [{elapsed:.1f}s]')
        except Exception as e:
            error_log.append({'label': label, 'error': str(e)})
            print(f'  {label:45s} → ERROR: {e}')

total_elapsed = time() - total_start
print()
print(f'총 train 데이터 {train_total}건, DSC {len(dsc_rows)}건 ({total_elapsed:.0f}초)')
if error_log:
    print(f'에러 {len(error_log)}건:')
    for e in error_log:
        print(f'  {e["label"]}: {e["error"]}')


데이터셋: TelcoCustomerChurn
  TelcoCustomerChurn/completeness_10%           → DSC= 95.17 (A)  [0.1s]
  TelcoCustomerChurn/uniqueness_10%             → DSC= 90.99 (A)  [0.2s]
  TelcoCustomerChurn/feature_accuracy_10%       → DSC= 97.65 (A)  [0.4s]


Will return maximum possible number of samples.


  TelcoCustomerChurn/consistent_repr_10%        → DSC= 96.25 (A)  [2.4s]
  TelcoCustomerChurn/class_balance_10%          → DSC= 99.48 (A)  [0.0s]
  TelcoCustomerChurn/completeness_25%           → DSC= 91.34 (A)  [0.0s]
  TelcoCustomerChurn/uniqueness_25%             → DSC= 87.65 (B)  [0.2s]
  TelcoCustomerChurn/feature_accuracy_25%       → DSC= 97.65 (A)  [0.6s]


Will return maximum possible number of samples.


  TelcoCustomerChurn/consistent_repr_25%        → DSC= 94.14 (A)  [8.9s]
  TelcoCustomerChurn/class_balance_25%          → DSC= 98.75 (A)  [0.0s]
  TelcoCustomerChurn/completeness_50%           → DSC= 84.79 (B)  [0.0s]
  TelcoCustomerChurn/uniqueness_50%             → DSC= 84.32 (B)  [0.5s]
  TelcoCustomerChurn/feature_accuracy_50%       → DSC= 97.62 (A)  [1.2s]


...

  letter/feature_accuracy_25%                   → DSC= 99.64 (A)  [0.0s]
  letter/class_balance_25%                      → DSC= 97.42 (A)  [0.1s]
  letter/completeness_50%                       → DSC= 87.25 (B)  [0.0s]
  letter/uniqueness_50%                         → DSC= 84.97 (B)  [1.4s]


Will return maximum possible number of samples.


  letter/feature_accuracy_50%                   → DSC= 99.68 (A)  [0.0s]
  letter/class_balance_50%                      → DSC= 96.19 (A)  [0.2s]
  letter/completeness_75%                       → DSC= 78.04 (B)  [0.1s]
  letter/uniqueness_75%                         → DSC= 83.31 (B)  [2.4s]


Will return maximum possible number of samples.


  letter/feature_accuracy_75%                   → DSC= 99.69 (A)  [0.0s]
  letter/class_balance_75%                      → DSC= 94.93 (A)  [0.1s]

총 59건 완료 (111초)


In [7]:
# ============================================================
# 2-3. DSC 점수 저장 (베이스라인 + 오염 전부)
# ============================================================
df_dsc = pd.DataFrame(dsc_rows)
dsc_path = f'{RESULTS_DIR}/dsc_scores.csv'
df_dsc.to_csv(dsc_path, index=False)

print(f'DSC 점수 저장: {dsc_path}')
print(f'총 {len(df_dsc)}건')
print(f'\n--- 노트북 02 완료 ---')
print(f'다음: 03_training.ipynb 실행')

df_dsc.head(10)

DSC 점수 저장: /content/drive/MyDrive/capstone/dsc/results/dsc_scores.csv
총 59건

--- 노트북 02 완료 ---
다음: 03_training.ipynb 실행


,dataset,polluter,level,score,grade,completeness,uniqueness,validity,consistency,outlier_ratio,class_balance,feature_correlation
0,TelcoCustomerChurn,none,0.00,97.65,A,1.0000,1.0000,0.9999,1.0000,1.0000,0.5307,1.0
1,SouthGermanCredit,none,0.00,97.45,A,1.0000,1.0000,1.0000,1.0000,0.9450,0.6000,1.0
2,letter,none,0.00,98.11,A,1.0000,0.9334,1.0000,1.0000,0.9671,0.9542,1.0
3,TelcoCustomerChurn,completeness,0.10,95.17,A,0.9050,1.0000,1.0000,0.9938,0.9985,0.5307,1.0
4,TelcoCustomerChurn,uniqueness,0.10,90.99,A,1.0000,0.6666,1.0000,1.0000,1.0000,0.5308,1.0
5,TelcoCustomerChurn,feature_accuracy,0.10,97.65,A,1.0000,1.0000,1.0000,1.0000,0.9997,0.5307,1.0
6,TelcoCustomerChurn,consistent_repr,0.10,96.25,A,1.0000,1.0000,1.0000,0.9063,1.0000,0.5307,1.0
7,TelcoCustomerChurn,class_balance,0.10,99.48,A,1.0000,1.0000,1.0000,1.0000,0.9981,0.9005,1.0
8,TelcoCustomerChurn,completeness,0.25,91.34,A,0.7626,1.0000,1.0000,0.9844,0.9854,0.5307,1.0
9,TelcoCustomerChurn,uniqueness,0.25,87.65,B,1.0000,0.5000,1.0000,1.0000,1.0000,0.5307,1.0


In [9]:
# ============================================================
# 3. 실행 로그 저장
# ============================================================
from datetime import datetime

log_lines = []
log_lines.append('# 노트북 02 실행 로그: 오염 생성 & DSC 점수 계산')
log_lines.append('')
log_lines.append(f'- **실행 시각**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'- **총 실험**: {len(dsc_rows)}건 (베이스라인 포함)')
log_lines.append(f'- **소요 시간**: {total_elapsed:.0f}초')
log_lines.append(f'- **에러**: {len(error_log)}건')
log_lines.append('')

# 오염 설정
log_lines.append('## 1. 오염 설정')
log_lines.append('')
log_lines.append(f'- **데이터셋**: {list(DATASETS.keys())}')
log_lines.append(f'- **오염 강도**: {POLLUTION_LEVELS}')
log_lines.append(f'- **Polluter**: CompletenessPolluter, UniquenessPolluter, FeatureAccuracyPolluter, ConsistentRepresentationPolluter, ClassBalancePolluter')
log_lines.append('')

# DSC 점수 전체 테이블
log_lines.append('## 2. DSC 점수 결과')
log_lines.append('')
log_lines.append('| 데이터셋 | 오염 유형 | 강도 | DSC Score | 등급 | completeness | uniqueness | validity | consistency | outlier_ratio | class_balance | feature_corr |')
log_lines.append('|---|---|---|---|---|---|---|---|---|---|---|---|')
for row in dsc_rows:
    log_lines.append(f'| {row["dataset"]} | {row["polluter"]} | {row["level"]} | {row["score"]} | {row["grade"]} | {row.get("completeness","")} | {row.get("uniqueness","")} | {row.get("validity","")} | {row.get("consistency","")} | {row.get("outlier_ratio","")} | {row.get("class_balance","")} | {row.get("feature_correlation","")} |')
log_lines.append('')

# 데이터셋별 DSC 변화 요약
log_lines.append('## 3. 오염 강도별 DSC 변화 요약')
log_lines.append('')
for ds_name in DATASETS:
    log_lines.append(f'### {ds_name}')
    log_lines.append('')
    log_lines.append('| 오염 유형 | 0%(baseline) | 10% | 25% | 50% | 75% | 최대 하락폭 |')
    log_lines.append('|---|---|---|---|---|---|---|')

    baseline_score = None
    for row in dsc_rows:
        if row['dataset'] == ds_name and row['polluter'] == 'none':
            baseline_score = row['score']
            break

    polluter_names = sorted(set(r['polluter'] for r in dsc_rows if r['dataset'] == ds_name and r['polluter'] != 'none'))
    for pname in polluter_names:
        scores_by_level = {0.0: baseline_score if baseline_score else '-'}
        for row in dsc_rows:
            if row['dataset'] == ds_name and row['polluter'] == pname:
                scores_by_level[row['level']] = row['score']

        s10 = scores_by_level.get(0.1, 'ERR')
        s25 = scores_by_level.get(0.25, 'ERR')
        s50 = scores_by_level.get(0.5, 'ERR')
        s75 = scores_by_level.get(0.75, 'ERR')

        vals = [v for v in [s10, s25, s50, s75] if isinstance(v, (int, float))]
        if vals and baseline_score:
            max_drop = round(baseline_score - min(vals), 2)
        else:
            max_drop = '-'

        log_lines.append(f'| {pname} | {baseline_score} | {s10} | {s25} | {s50} | {s75} | {max_drop} |')
    log_lines.append('')

# 에러 로그
if error_log:
    log_lines.append('## 4. 에러 목록')
    log_lines.append('')
    for e in error_log:
        err_msg = str(e.get('error', 'unknown'))
        if len(err_msg) > 200:
            err_msg = err_msg[:200] + '...'
        log_lines.append(f'- **{e["label"]}**: {err_msg}')
    log_lines.append('')

# 문제점 분석 (DSC 점수 변화가 작은 경우 자동 감지)
log_lines.append('## 5. 자동 감지된 문제점')
log_lines.append('')
for ds_name in DATASETS:
    baseline_s = None
    for row in dsc_rows:
        if row['dataset'] == ds_name and row['polluter'] == 'none':
            baseline_s = row['score']
            break
    if baseline_s is None:
        continue
    for row in dsc_rows:
        if row['dataset'] == ds_name and row['level'] == 0.75 and row['polluter'] != 'none':
            drop = baseline_s - row['score']
            if drop < 5:
                log_lines.append(f'- ⚠️ **{ds_name}/{row["polluter"]}_75%**: DSC 하락 {drop:.1f}점 (5점 미만 — 오염 감지 부족)')
log_lines.append('')

# 산출물
log_lines.append('## 6. 산출물')
log_lines.append('')
log_lines.append(f'- `dsc_scores.csv` — DSC 점수 {len(dsc_rows)}건')
log_lines.append(f'- `data/polluted/` — 오염 데이터 파일')
log_lines.append(f'- `02_execution_log.md` — 이 로그 파일')
log_lines.append('')
log_lines.append('---')
log_lines.append('*이 로그는 노트북 02 실행 시 자동 생성됨*')

log_path = f'{RESULTS_DIR}/02_execution_log.md'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print(f'실행 로그 저장: {log_path}')


실행 로그 저장: /content/drive/MyDrive/capstone/dsc/results/02_execution_log.md
